In [3]:
import sys
import os
sys.path.append("../")
sys.path.append("../../")

In [4]:
from scale_rl.common.wandb_utils import *

In [9]:
entity = 'draftrec'
project_name = 'Simba_2412'
run_exp_names_to_analysis_exp_names = {
    'sac_simba': 'sac_simba',
}
run_exp_names = list(run_exp_names_to_analysis_exp_names.keys())
metrics = ['avg_return', 'avg_success']

In [ ]:
runs = collect_runs(entity=entity, project_name=project_name) 
filtered_runs = filter_runs(runs, exp_names = run_exp_names)
wandb_df = convert_runs_to_dataframe(
    runs = filtered_runs, 
    run_exp_name_to_analysis_exp_name=run_exp_names_to_analysis_exp_names
)
wandb_df = wandb_df[wandb_df.apply(lambda row: 'finished' in str(row['run']), axis=1)]
eval_df = convert_wandb_df_to_eval_df(wandb_df, metrics)
eval_df


  0%|          | 0/260 [00:00<?, ?it/s]

Exception ignored in: <function tqdm.__del__ at 0x7f9db731a8b0>
Traceback (most recent call last):
  File "/home/nas4_user/youngdolee/anaconda3/envs/rl_playground/lib/python3.9/site-packages/tqdm/std.py", line 1149, in __del__
    self.close()
  File "/home/nas4_user/youngdolee/anaconda3/envs/rl_playground/lib/python3.9/site-packages/tqdm/notebook.py", line 272, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'


  0%|          | 0/260 [00:00<?, ?it/s]

  0%|          | 0/260 [00:00<?, ?it/s]

In [ ]:
def update_seeds_without_value_check(eval_df):
    """
    Updates the seed values in eval_df when exp_name, env_name, and env_step are the same.

    Args:
    - eval_df (pandas DataFrame): DataFrame containing the evaluation data.

    Returns:
    - eval_df (pandas DataFrame): Updated DataFrame with non-duplicate seed values for 
                                  matching exp_name, env_name, and env_step.
    """
    # Create a copy of the DataFrame to avoid modifying the original
    updated_df = eval_df.copy()

    # Track the seeds that have been used for each (exp_name, env_name, env_step)
    seed_tracker = {}

    # Iterate over the DataFrame row by row
    for idx, row in updated_df.iterrows():
        exp_name = row["exp_name"]
        env_name = row["env_name"]
        env_step = row["env_step"]
        metric = row['metric']
        seed = row["seed"]

        key = (exp_name, env_name, env_step, metric)

        if key not in seed_tracker:
            seed_tracker[key] = set()

        # If the current seed has already been used for this key (exp_name, env_name, env_step)
        if seed in seed_tracker[key]:
            # Increment the seed until we find a unique one
            new_seed = seed + 500
            while new_seed in seed_tracker[key]:
                new_seed += 500
            # Update the seed in the DataFrame
            updated_df.at[idx, "seed"] = new_seed
            # Add the new seed to the tracker
            seed_tracker[key].add(new_seed)
        else:
            # Add the current seed to the tracker
            seed_tracker[key].add(seed)

    return updated_df

In [ ]:
new_df = update_seeds_without_value_check(eval_df)

In [ ]:
new_df.to_csv("/home/nas4_user/youngdolee/SimbaV2/results/simba_dmc_mujoco_1m.csv")